# BinX AI & ML Internship
## Week 5 — Day 3: Dimensionality Reduction with PCA

**Topics covered:**
- The curse of dimensionality — why too many features is a real problem
- What PCA does: principal components and variance
- Explained variance ratio
- Choosing the number of components (the 95% rule)
- When (and when not) to use PCA

---
## 1. Where I'm Starting From

Days 1 and 2 were about clustering — finding groups in the data.  
K-Means, DBSCAN, and hierarchical clustering all work on the
feature space as-is: they compute distances between points.

But what happens when a dataset has many features?

```text
Insurance dataset (encoded):
  age, bmi, children, charges,
  sex_male, smoker_yes,
  region_northwest, region_southeast, region_southwest
  → 9 dimensions total

Real ML datasets can have hundreds or thousands of features.

The distance between points in 9-dimensional space is already
harder to visualize and reason about.
In 500 dimensions, the concept of distance becomes unreliable.
```

Dimensionality reduction compresses many features into a few
while keeping as much information as possible.

Today I learn the most widely used method: **PCA**.

---
## 2. The Curse of Dimensionality

High-dimensional data causes real, measurable problems:

**Problem 1 — Distances lose meaning:**

In low dimensions, distance reliably tells us which points are close and which are far.  
In very high dimensions, all distances tend to become similar —  
every point appears to be roughly the same distance from every other point.  
This breaks any distance-based method, including K-Means.

**Problem 2 — Data becomes sparse:**

```text
1D:  100 points fill a line reasonably well.
2D:  100 points in a square — starts to feel sparse.
3D:  100 points in a cube — clearly sparse.
10D: 100 points in a 10-dimensional hypercube — essentially empty.
```

To cover the same density in higher dimensions, you need exponentially more data.

**Problem 3 — Models overfit more easily:**

More features means more parameters.  
When the number of features approaches or exceeds the number of observations,  
models can fit the training set perfectly while generalizing poorly to new data.

**Problem 4 — You cannot visualize it:**

The human brain handles 2D and 3D well. Beyond that, direct visualization is impossible.

**The solution:** Reduce the number of features while retaining as much information as possible.  
This is what PCA does.

---
## 3. What PCA Does

**Principal Component Analysis (PCA)** finds new axes — called **principal components** —  
that capture the directions of greatest variance in the data.

```text
Original data → many (possibly correlated) features
PCA finds     → uncorrelated axes ordered by how much variance they capture
                → PC1 captures the most variance
                → PC2 captures the second most (at a right angle to PC1)
                → ...and so on
```

By keeping only the first few components, I reduce the number of dimensions  
while keeping most of the information.

---

### The geometry (2D example)

```text
Original features:
  x1 (bmi)     ↑
                │    ●●
                │  ●●●●
                │ ●●●
                └──────── x2 (charges)

The data has a main direction (upward-right):
points with higher bmi tend to have higher charges.

PCA finds this direction and calls it PC1.
It then finds the next most important direction
(at 90° to PC1) and calls it PC2.
```

---

### Why "principal"?

The components are ranked by variance:

```text
PC1 → captures the most variance in the data
PC2 → captures the second most variance (perpendicular to PC1)
PC3 → captures the third most, etc.
```

This ordering means I can drop the last components (which capture very little variance)  
and lose only a small amount of information.

---

### What PCA actually computes

Each principal component is a **weighted combination** of the original features:

```text
PC1 = w1·age + w2·bmi + w3·children + w4·charges + ...
```

The weights (called **loadings**) tell me which original features contribute most to each component.  
This is how I can interpret what PC1 or PC2 "represents".

In [2]:
# The PCA API — how to use it
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np

# PCA always requires scaled data first
# X_scaled = StandardScaler().fit_transform(X)

# Fit PCA with all components
# pca = PCA()                           # no n_components → keeps all
# pca.fit(X_scaled)

# Each component's contribution
# pca.explained_variance_ratio_         # e.g. [0.22, 0.16, 0.15, ...]
# np.cumsum(pca.explained_variance_ratio_)  # cumulative sum

# Reduce to 2D
# pca2 = PCA(n_components=2)
# X_reduced = pca2.fit_transform(X_scaled)  # shape: (n_samples, 2)

print("PCA API overview — see Hands-On Lab for live results.")

PCA API overview — see Hands-On Lab for live results.


---
## 4. Explained Variance Ratio

The **explained variance ratio** answers:  
> *"What fraction of the total information in the data does each component carry?"*

```text
pca.explained_variance_ratio_
→ [0.2156, 0.1605, 0.1471, 0.1216, 0.1120, 0.1061, 0.0874, 0.0350, 0.0148]

PC1 keeps 21.6% of the total information
PC2 keeps 16.0%
PC1 + PC2 together: 37.6%
```

Taking the cumulative sum shows when a target threshold is reached:

```text
n=1  →  21.6%
n=2  →  37.6%
n=3  →  52.3%
n=4  →  64.5%
n=5  →  75.7%
n=6  →  86.3%
n=7  →  95.0%  ← 95% threshold crossed
n=8  →  98.5%
n=9  → 100.0%
```

---

### The 95% rule

A widely used guideline is to keep enough components to retain **~95% of the variance**.

```text
9 features → 7 components
5% of information discarded
Models trained on 7 components behave almost identically to models on all 9
```

The exact threshold (90%, 95%, 99%) depends on the use case:  
- For speed and storage: 90% may be enough  
- For modelling where small differences matter: 99% is safer

---
## 5. Choosing the Number of Components

Three approaches:

**Method 1 — Cumulative Explained Variance Plot**

```python
import numpy as np
import matplotlib.pyplot as plt

cumulative = np.cumsum(pca.explained_variance_ratio_)
plt.plot(range(1, len(cumulative)+1), cumulative, 'o-')
plt.axhline(0.95, linestyle='--', color='green', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
```

**Method 2 — Direct calculation**

```python
cumulative = np.cumsum(pca.explained_variance_ratio_)
n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"Components needed: {n_95}")
```

**Method 3 — Pass threshold directly to PCA**

```python
pca_95 = PCA(n_components=0.95)   # sklearn handles the threshold automatically
X_95 = pca_95.fit_transform(X_scaled)
print(f"Components kept: {pca_95.n_components_}")
```

All three approaches give the same result — choose the one that fits the workflow.

---
## 6. When (and When Not) to Use PCA

### Use PCA when:

| Situation | Why PCA helps |
|-----------|---------------|
| Many correlated features | PCA combines correlated columns into one component |
| Model is overfitting | Removing low-variance components reduces noise |
| Slow training | Fewer features → faster training and prediction |
| Visualization needed | Reduce to 2D or 3D to plot otherwise invisible data |

### Do NOT use PCA when:

| Situation | Reason |
|-----------|--------|
| Feature interpretability is needed | PCA components are combinations — you lose the ability to say "bmi alone drove this" |
| Features are already independent | No correlation to compress → PCA won't help |
| Dataset is small | Adds complexity without benefit |
| Exact reconstruction needed | Reducing dimensions always loses some information |

---

### The key trade-off

```text
Fewer components  →  faster, simpler, easier to visualize
                  →  BUT: some information is lost

More components   →  more information retained
                  →  BUT: closer to the original high-dimensional space
```

PCA does not know which features are important for *prediction* —  
it only knows which directions have the most *variance*.  
A feature that strongly predicts the target but has low variance  
may end up compressed into a late, low-importance component.

This is why PCA is applied **before modelling** as a preprocessing step,  
not as a replacement for feature selection.

---

### Summary

```text
Curse of dimensionality → too many features hurts models and visualization
PCA                     → finds directions of maximum variance (principal components)
Explained variance ratio → how much information each component carries
95% rule                → keep enough components to retain ~95% of variance
2D reduction            → allows visualization; retains 37.6% of insurance variance
```

The practical application is in `Hands_On_Lab.ipynb`.